# Mamba SOH — Training at L=4096 on Kaggle GPU

Train MambaSOHPredictor with 4096-token windows on Kaggle P100/T4 GPU.

**Before running:** Add dataset `nasa-battery-dataset` via `+ Add Data`.

Dataset must contain: `cleaned_dataset/metadata.csv` and `cleaned_dataset/data/*.csv`

## Cell 1 — GPU Check

In [ ]:
!nvidia-smi

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f'VRAM: {vram} GB')
else:
    print('WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU P100')

## Cell 2 — Clone Repo

Change `GITHUB_TOKEN` secret name if different, or use public clone below.

In [ ]:
import subprocess, os

BRANCH = 'feat/spectral_kurtosis'
REPO   = '/kaggle/working/ai-module'

# --- Option A: Public repo ---
# subprocess.run([
#     'git', 'clone', '--branch', BRANCH, '--single-branch',
#     'https://github.com/GSU26SE55/ai-module.git', REPO
# ], check=True)

# --- Option B: Private repo (uses Kaggle Secret) ---
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        f'https://{token}@github.com/GSU26SE55/ai-module.git', REPO
    ], check=True)
    # Remove token from remote URL
    subprocess.run([
        'git', '-C', REPO, 'remote', 'set-url', 'origin',
        'https://github.com/GSU26SE55/ai-module.git'
    ], check=True)
    print('Cloned with token (token cleared from remote)')
except Exception as e:
    print(f'Secret not found ({e}), trying public clone...')
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/GSU26SE55/ai-module.git', REPO
    ], check=True)

print('Branch:', subprocess.check_output(['git', '-C', REPO, 'branch', '--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--oneline']).decode().strip())

## Cell 3 — Install Dependencies

In [ ]:
%pip install -q scipy scikit-learn

import scipy, sklearn
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)

## Cell 4 — Path Setup & Dataset Check

In [ ]:
import os, sys, shutil

REPO      = '/kaggle/working/ai-module'
PROCESSED = '/kaggle/working/processed'
WEIGHTS   = '/kaggle/working/weights'
LOGS      = '/kaggle/working/logs'

# Auto-detect dataset path
for candidate in [
    '/kaggle/input/nasa-battery-dataset/cleaned_dataset',
    '/kaggle/input/nasa-battery-dataset',
]:
    if os.path.isfile(f'{candidate}/metadata.csv'):
        DATASET = candidate
        break
else:
    # Search
    result = subprocess.check_output(['find', '/kaggle/input', '-name', 'metadata.csv']).decode().strip()
    if result:
        DATASET = os.path.dirname(result.split('\n')[0])
    else:
        raise FileNotFoundError('metadata.csv not found. Add nasa-battery-dataset via + Add Data')

for path in [PROCESSED, WEIGHTS, LOGS]:
    os.makedirs(path, exist_ok=True)

# Symlink repo weights -> WEIGHTS so training artifacts persist in /kaggle/working
repo_weights = f'{REPO}/models/weights'
if os.path.islink(repo_weights):
    os.unlink(repo_weights)
elif os.path.isdir(repo_weights):
    # Copy existing artifacts to WEIGHTS first
    for f in os.listdir(repo_weights):
        src = os.path.join(repo_weights, f)
        dst = os.path.join(WEIGHTS, f)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    shutil.rmtree(repo_weights)
os.makedirs(os.path.dirname(repo_weights), exist_ok=True)
os.symlink(WEIGHTS, repo_weights)

sys.path.insert(0, REPO)

print(f'REPO:      {REPO}')
print(f'DATASET:   {DATASET}')
print(f'PROCESSED: {PROCESSED}')
print(f'WEIGHTS:   {WEIGHTS}')
print()
print('metadata.csv:', os.path.isfile(f'{DATASET}/metadata.csv'))
data_dir = f'{DATASET}/data'
print('data/:', os.path.isdir(data_dir))
if os.path.isdir(data_dir):
    csvs = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    print(f'CSV count: {len(csvs)}')

## Cell 5 — Update Config to L=4096

In [ ]:
import os, subprocess

config_path = f'{REPO}/src/core/config.py'

# Ghi thẳng config — không dùng string replacement (dễ fail nếu format khác)
CONFIG_CONTENT = '''import os

MODEL_VERSION          = "1.3"
SCALER_VERSION         = "1.0"
FEATURE_SCALER_VERSION = "1.1"

BASE_DIR    = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
WEIGHTS_DIR = os.path.join(BASE_DIR, "models", "weights")

SCALER_PATH         = os.path.join(WEIGHTS_DIR, "scaler.pkl")
FEATURE_SCALER_PATH = os.path.join(WEIGHTS_DIR, "feature_scaler.pkl")
MAMBA_PATH          = os.path.join(WEIGHTS_DIR, f"soh_mamba_v{MODEL_VERSION}.pth")
ISO_FOREST_PATH     = os.path.join(WEIGHTS_DIR, f"isolation_forest_v{MODEL_VERSION}.pkl")

WINDOW_SIZE   = 4096
WINDOW_STRIDE = 100
INPUT_FEATURES    = 6
SPECTRAL_FEAT_DIM = 54
D_MODEL = 64
D_STATE = 16

FEATURES = [
    "voltage", "current", "temperature",
    "current_load", "voltage_load", "time",
]
RAW_FEATURES = [
    "Voltage_measured", "Current_measured", "Temperature_measured",
    "Current_load", "Voltage_load", "Time",
]

SEED = 42
'''

with open(config_path, 'w') as f:
    f.write(CONFIG_CONTENT)

# Verify bằng subprocess — tránh Python module cache
result = subprocess.check_output([
    'python', '-c',
    f'import sys; sys.path.insert(0, "{REPO}"); '
    'from src.core.config import WINDOW_SIZE, WINDOW_STRIDE, MODEL_VERSION; '
    'print(WINDOW_SIZE, WINDOW_STRIDE, MODEL_VERSION)'
]).decode().strip()

ws, stride, ver = result.split()
print(f'WINDOW_SIZE   = {ws}')
print(f'WINDOW_STRIDE = {stride}')
print(f'MODEL_VERSION = {ver}')
assert ws == '4096', f'Config patch failed! WINDOW_SIZE={ws}'
print('Config OK — ready to preprocess')

## Cell 6 — Preprocess

Skip if `/kaggle/working/processed/train.pt` already exists with correct shape.

In [ ]:
preprocess_path = f'{REPO}/scripts/preprocess.py'

# Ghi lại preprocess.py với long-sequence strategy (concatenate cycles)
PREPROCESS_CONTENT = '''"""
Preprocessing — long-sequence mode for L=4096.

NASA cycles are ~285 steps each, shorter than WINDOW_SIZE=4096.
Solution: concatenate ALL discharge cycles of a battery into one long
time-series (50,000+ steps), then slide a WINDOW_SIZE window with WINDOW_STRIDE.

Cycle-level FiLM features: computed on WINDOW_SIZE-step window (4096 pts → 2048 FFT bins).
"""
import argparse, os, random, sys
import joblib, numpy as np, pandas as pd, torch
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from src.core.config import FEATURE_SCALER_PATH, FEATURES, RAW_FEATURES, WINDOW_SIZE, WINDOW_STRIDE
from src.features.extractor import extract_window_features

SEED = 42
NOMINAL_CAPACITY = 2.0
TRAIN_IDS = ["B0005", "B0006", "B0007"]
VAL_IDS   = ["B0018"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def load_battery_sequence(data_dir, battery_id):
    """Concatenate all discharge cycles → one long (T, 6) array + per-timestep SOH."""
    meta  = pd.read_csv(os.path.join(data_dir, "metadata.csv"))
    cdir  = os.path.join(data_dir, "data")
    disch = (meta[(meta.battery_id == battery_id) & (meta.type == "discharge") & meta.Capacity.notna()]
             .sort_values("test_id").reset_index(drop=True))
    if len(disch) == 0:
        raise ValueError(f"No discharge cycles for {battery_id}")

    all_data, all_soh = [], []
    for _, row in disch.iterrows():
        path = os.path.join(cdir, row.filename)
        if not os.path.exists(path): continue
        df = pd.read_csv(path)
        if not all(c in df.columns for c in RAW_FEATURES): continue
        n = min(len(df[c]) for c in RAW_FEATURES)
        chunk = np.stack([df[c].values[:n].astype(np.float32) for c in RAW_FEATURES], axis=1)
        soh   = float(row.Capacity) / NOMINAL_CAPACITY * 100
        all_data.append(chunk)
        all_soh.append(np.full(n, soh, dtype=np.float32))

    return np.vstack(all_data), np.concatenate(all_soh)


def collect_batteries(data_dir, ids):
    seqs = {}
    for bid in ids:
        data, sohs = load_battery_sequence(data_dir, bid)
        seqs[bid] = (data, sohs)
        print(f"  {bid}: {len(data):,} timesteps, SOH [{sohs.min():.1f}–{sohs.max():.1f}%]")
    return seqs


def make_windows(data, sohs, scaler, feat_scaler=None):
    """Scale, extract features, slide window."""
    scaled = scaler.transform(data).astype(np.float32)
    T = len(scaled)
    Xs, Xf, ys = [], [], []
    for start in range(0, T - WINDOW_SIZE + 1, WINDOW_STRIDE):
        end = start + WINDOW_SIZE
        w   = scaled[start:end]
        Xs.append(w)
        Xf.append(extract_window_features(w[:, :3]))  # FiLM: voltage,current,temp
        ys.append(sohs[end - 1])
    X      = np.array(Xs, dtype=np.float32)
    X_feat = np.array(Xf, dtype=np.float32)
    y      = np.array(ys,  dtype=np.float32)
    if feat_scaler is not None:
        X_feat = feat_scaler.transform(X_feat).astype(np.float32)
    return X, X_feat, y


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir",   default="data/raw/nasa/cleaned_dataset")
    parser.add_argument("--output-dir", default="data/processed")
    args = parser.parse_args()
    os.makedirs(args.output_dir, exist_ok=True)

    print(f"Window size: {WINDOW_SIZE} | Stride: {WINDOW_STRIDE}")

    print("\\nLoading train batteries...")
    train_seqs = collect_batteries(args.data_dir, TRAIN_IDS)
    print("Loading val/test (B0018)...")
    b0018_seqs = collect_batteries(args.data_dir, VAL_IDS)

    b0018_data, b0018_sohs = list(b0018_seqs.values())[0]
    split = int(len(b0018_data) * 0.7)
    val_data  = (b0018_data[:split],  b0018_sohs[:split])
    test_data = (b0018_data[split:],  b0018_sohs[split:])

    # Fit MinMaxScaler on all train timesteps
    print("\\nFitting MinMaxScaler...")
    train_raw = np.vstack([d for d, _ in train_seqs.values()])
    scaler    = MinMaxScaler()
    scaler.fit(train_raw)
    os.makedirs("models/weights", exist_ok=True)
    joblib.dump({"scaler": scaler, "version": "1.0", "trained_on": TRAIN_IDS, "features": FEATURES},
                "models/weights/scaler.pkl")
    print("Saved scaler")

    # Extract windows + raw features for train
    print("\\nExtracting windows and features...")
    Xp, Xfp, yp = [], [], []
    for bid, (data, sohs) in train_seqs.items():
        X, Xf, y = make_windows(data, sohs, scaler)
        Xp.append(X); Xfp.append(Xf); yp.append(y)
        print(f"  {bid}: {len(X)} windows")
    X_train      = np.vstack(Xp)
    X_feat_train = np.vstack(Xfp)
    y_train      = np.concatenate(yp)

    feat_scaler  = StandardScaler()
    X_feat_train = feat_scaler.fit_transform(X_feat_train).astype(np.float32)
    os.makedirs(os.path.dirname(FEATURE_SCALER_PATH), exist_ok=True)
    joblib.dump({"scaler": feat_scaler, "version": "1.1", "n_features": X_feat_train.shape[1]},
                FEATURE_SCALER_PATH)
    print(f"Saved feature_scaler (n_features={X_feat_train.shape[1]})")

    X_val,  Xf_val,  y_val  = make_windows(*val_data,  scaler, feat_scaler)
    X_test, Xf_test, y_test = make_windows(*test_data, scaler, feat_scaler)

    print(f"\\nSplit summary:")
    print(f"  Train: {len(X_train):>5} windows   feat={X_feat_train.shape[1]}")
    print(f"  Val  : {len(X_val):>5} windows")
    print(f"  Test : {len(X_test):>5} windows")

    if len(X_test) == 0:
        raise RuntimeError(f"Test set empty — B0018 last 30% has fewer than {WINDOW_SIZE} timesteps")

    for name, X, Xf, y in [
        ("train", X_train,  X_feat_train, y_train),
        ("val",   X_val,    Xf_val,       y_val),
        ("test",  X_test,   Xf_test,      y_test),
    ]:
        torch.save(
            {"X": torch.tensor(X), "X_feat": torch.tensor(Xf), "y": torch.tensor(y),
             "feature_scaler_version": "1.1"},
            os.path.join(args.output_dir, f"{name}.pt"),
        )
        print(f"Saved {name}.pt ({len(X)} samples)")

    print("\\nPreprocessing complete.")


if __name__ == "__main__":
    main()
'''

with open(preprocess_path, 'w') as f:
    f.write(PREPROCESS_CONTENT)

print('preprocess.py patched for long-sequence (L=4096)')
print(f'NASA cycles ~285 steps → concatenate all cycles per battery → slide L=4096 window')
print(f'Expected: ~1,400 train windows (50,000 timesteps / stride 100)')

## Cell 5b — Patch preprocess.py for L=4096

NASA cycles ~285 steps < 4096. Must concatenate all cycles per battery into one long sequence, then slide L=4096 window.

In [ ]:
import torch

# Check if processed data already exists
need_preprocess = True
train_pt = f'{PROCESSED}/train.pt'
if os.path.isfile(train_pt):
    d = torch.load(train_pt, weights_only=False)
    if d['X'].shape[1] == 4096:
        print(f'Processed data exists with correct L=4096. Skipping preprocess.')
        print(f'Train: {d["X"].shape}')
        need_preprocess = False
    else:
        print(f'Old data shape {d["X"].shape[1]} != 4096. Re-preprocessing...')

if need_preprocess:
    os.chdir(REPO)
    !python scripts/preprocess.py \
        --data-dir "{DATASET}" \
        --output-dir "{PROCESSED}"

## Cell 7 — Verify Data Shape

In [ ]:
import os, subprocess, inspect

# ─── 1. Verify extractor is ORIGINAL version ────────────────────────────────
extractor_path = f'{REPO}/src/features/extractor.py'
with open(extractor_path) as f:
    ext_src = f.read()

if 'polyfit' in ext_src or 'hanning' in ext_src or 'spectral_slope' in ext_src:
    print("❌ MODIFIED extractor detected — rewriting to original...")
    ORIGINAL_EXTRACTOR = '''"""
Feature extraction: Spectral + Statistical (Kurtosis) features.
Each window (T, C) yields 54 scalar features (9 spectral + 9 statistical × 3 channels).
"""
import numpy as np
from scipy.stats import kurtosis as scipy_kurtosis
from scipy.stats import skew as scipy_skew


def _spectral_features(x: np.ndarray) -> np.ndarray:
    n = len(x)
    fft_vals = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(n, d=1.0)
    power = np.abs(fft_vals) ** 2
    total = power.sum()
    if total < 1e-12:
        power = power + 1e-12; total = power.sum()
    p_norm = power / total
    centroid = float(np.dot(freqs, p_norm))
    log_len = np.log(len(p_norm) + 1e-12)
    entropy = float(-np.sum(p_norm * np.log(p_norm + 1e-12)) / log_len)
    peak_idx = int(np.argmax(power))
    peak_freq = float(freqs[peak_idx])
    peak_power_db = float(10.0 * np.log10(power[peak_idx] + 1e-12))
    log_mean = np.mean(np.log(power + 1e-12))
    arith_mean = np.mean(power)
    flatness = float(np.exp(log_mean) / (arith_mean + 1e-12))
    cumsum = np.cumsum(power)
    rolloff_idx = int(np.searchsorted(cumsum, 0.85 * total))
    rolloff_idx = min(rolloff_idx, len(freqs) - 1)
    rolloff = float(freqs[rolloff_idx])
    bs = len(power) // 3
    band_low  = float(power[:bs].sum() / total)
    band_mid  = float(power[bs:2*bs].sum() / total)
    band_high = float(power[2*bs:].sum() / total)
    return np.array([centroid, entropy, peak_freq, peak_power_db,
                     flatness, rolloff, band_low, band_mid, band_high], dtype=np.float32)


def _statistical_features(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float64)
    mean_val = float(np.mean(x)); std_val = float(np.std(x, ddof=0))
    if std_val < 1e-8:
        skew_val = 0.0; kurt_val = 0.0
    else:
        skew_val = float(scipy_skew(x, bias=False))
        kurt_val = float(scipy_kurtosis(x, fisher=True, bias=False))
    rms = float(np.sqrt(np.mean(x**2)))
    abs_max = float(np.max(np.abs(x))); mean_abs = float(np.mean(np.abs(x))) + 1e-12
    crest_factor = abs_max / (rms + 1e-12); waveform_factor = rms / mean_abs
    pulse_factor = abs_max / mean_abs
    margin_factor = abs_max / (float(np.mean(np.sqrt(np.abs(x))))**2 + 1e-12)
    ptp = float(np.ptp(x))
    return np.array([mean_val, std_val, skew_val, kurt_val, crest_factor,
                     waveform_factor, pulse_factor, margin_factor, ptp], dtype=np.float32)


def extract_window_features(window: np.ndarray) -> np.ndarray:
    if window.ndim != 2:
        raise ValueError(f"Expected (T, C), got {window.shape}")
    n_ch = window.shape[1]
    return np.concatenate(
        [_spectral_features(window[:, c]) for c in range(n_ch)] +
        [_statistical_features(window[:, c]) for c in range(n_ch)],
        dtype=np.float32,
    )

def extract_batch_features(windows: np.ndarray) -> np.ndarray:
    return np.stack([extract_window_features(w) for w in windows], axis=0)
'''
    with open(extractor_path, 'w') as f:
        f.write(ORIGINAL_EXTRACTOR)
    print("✅ Original extractor restored")
else:
    print("✅ Extractor is original version")

# ─── 2. Update config: stride=30, d_model=128 ───────────────────────────────
config_path = f'{REPO}/src/core/config.py'
CONFIG = '''import os

MODEL_VERSION          = "1.3"
SCALER_VERSION         = "1.0"
FEATURE_SCALER_VERSION = "1.1"

BASE_DIR    = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
WEIGHTS_DIR = os.path.join(BASE_DIR, "models", "weights")

SCALER_PATH         = os.path.join(WEIGHTS_DIR, "scaler.pkl")
FEATURE_SCALER_PATH = os.path.join(WEIGHTS_DIR, "feature_scaler.pkl")
MAMBA_PATH          = os.path.join(WEIGHTS_DIR, f"soh_mamba_v{MODEL_VERSION}.pth")
ISO_FOREST_PATH     = os.path.join(WEIGHTS_DIR, f"isolation_forest_v{MODEL_VERSION}.pkl")

WINDOW_SIZE   = 4096
WINDOW_STRIDE = 30    # stride=30 → ~4,500 train windows (same density as L=30 best run)
INPUT_FEATURES    = 6
SPECTRAL_FEAT_DIM = 54
D_MODEL = 128         # 64→128: more capacity for 4096-token context
D_STATE = 16

FEATURES = ["voltage","current","temperature","current_load","voltage_load","time"]
RAW_FEATURES = ["Voltage_measured","Current_measured","Temperature_measured",
                "Current_load","Voltage_load","Time"]
SEED = 42
'''
with open(config_path, 'w') as f:
    f.write(CONFIG)
print("✅ Config: WINDOW_STRIDE=30, D_MODEL=128")

# ─── 3. Patch soh_predictor.py: add JIT scan for GPU speed ──────────────────
predictor_path = f'{REPO}/src/models/soh_predictor.py'
with open(predictor_path) as f:
    pred_src = f.read()

if '_jit_scan' not in pred_src:
    # Add JIT-compiled scan at module level (runs in C++ on GPU)
    jit_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F


@torch.jit.script
def _jit_scan(dA: torch.Tensor, dBx: torch.Tensor) -> torch.Tensor:
    """JIT-compiled sequential SSM scan — C++ speed, no Python loop overhead."""
    B, L, D, N = dA.shape
    out = torch.empty_like(dBx)
    h = torch.zeros(B, D, N, device=dA.device, dtype=dA.dtype)
    for t in range(L):
        h = dA[:, t] * h + dBx[:, t]
        out[:, t] = h
    return out

'''
    pred_src = pred_src.replace(
        'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n',
        jit_code
    )
    # Replace inline sequential scan with JIT call
    pred_src = pred_src.replace(
        '            h = torch.zeros(B, d_inner, self.d_state, device=x.device, dtype=x.dtype)\n'
        '            ys = []\n'
        '            for t in range(L):\n'
        '                h = dA[:, t] * h + dBx[:, t]\n'
        '                y_t = (h * C_proj[:, t].unsqueeze(1)).sum(-1)\n'
        '                ys.append(y_t)\n'
        '            y = torch.stack(ys, dim=1)',
        '            h_out = _jit_scan(dA, dBx)  # JIT: C++ speed on GPU\n'
        '            y = (h_out * C_proj.unsqueeze(2)).sum(-1)'
    )
    with open(predictor_path, 'w') as f:
        f.write(pred_src)
    print("✅ JIT scan added to soh_predictor.py")
else:
    print("✅ JIT scan already present")

# ─── 4. Patch train.py: patience=25, scheduler patience=10, batch=4 ─────────
train_path = f'{REPO}/scripts/train.py'
with open(train_path) as f:
    train_src = f.read()

# Update constants
for old, new in [
    ('PATIENCE       = 15', 'PATIENCE       = 25'),
    ('BATCH_SIZE     = 32', 'BATCH_SIZE     = 4'),   # d_model=128 needs smaller batch
    ('BATCH_SIZE = 32',     'BATCH_SIZE = 4'),
    ('patience=5, min_lr=1e-6', 'patience=10, min_lr=1e-6'),
    ('BATCH_SIZE     = 1', 'BATCH_SIZE     = 4'),
    ('accumulation-steps', 'accum-steps-unused'),    # disable accum in args if present
]:
    train_src = train_src.replace(old, new)

with open(train_path, 'w') as f:
    f.write(train_src)
print("✅ train.py: PATIENCE=25, BATCH=4, scheduler patience=10")

# ─── 5. Verify via subprocess ────────────────────────────────────────────────
result = subprocess.check_output([
    'python', '-c',
    f'import sys; sys.path.insert(0,"{REPO}"); '
    'from src.core.config import WINDOW_SIZE,WINDOW_STRIDE,D_MODEL; '
    'print(WINDOW_SIZE, WINDOW_STRIDE, D_MODEL)'
]).decode().strip()
ws, stride, dm = result.split()
print(f"\nConfig verified: L={ws}, stride={stride}, d_model={dm}")
assert ws == '4096' and stride == '30' and dm == '128'
print("\n✅ All optimizations applied. Next: re-run preprocess then train.")

## Cell 7c — Full Optimization for MAE < 1%

Applies all improvements before training:
1. stride=30 → ~4,500 train samples (same as L=30 best run)
2. d_model=128 → more capacity for 4096-token context  
3. JIT scan → 10-30× faster than Python loop on GPU
4. Patience=25, scheduler patience=10 → give model time to converge
5. Verify extractor is original version

## Cell 7b — Patch soh_predictor.py for GPU Speed

Sequential scan with L=4096 = 4096 Python loop iterations per batch → too slow.
Fix: JIT-compile the scan loop → 10-50× faster on GPU.

In [ ]:
import torch

for name in ['train.pt', 'val.pt', 'test.pt']:
    d = torch.load(f'{PROCESSED}/{name}', weights_only=False)
    x_shape  = tuple(d['X'].shape)
    xf_shape = tuple(d['X_feat'].shape)
    y_range  = (float(d['y'].min()), float(d['y'].max()))
    print(f'{name}: X={x_shape}, X_feat={xf_shape}, y=[{y_range[0]:.1f},{y_range[1]:.1f}]')
    assert x_shape[1] == 4096, f'Wrong window size: {x_shape[1]}'
    assert xf_shape[1] == 54,  f'Wrong feat dim: {xf_shape[1]}'

print('\nAll shapes OK')

## Cell 8 — Add GPU Support to train.py

The simple train.py runs on CPU only. This cell patches it to use GPU automatically.

In [ ]:
train_path = f'{REPO}/scripts/train.py'

with open(train_path) as f:
    code = f.read()

# Check if GPU support already added
if 'device = torch.device' not in code:
    # Add device after model creation
    code = code.replace(
        'torch.manual_seed(SEED)\n    model     = MambaSOHPredictor(',
        'device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
        '    logger.info(f"Device: {device}")\n'
        '    torch.manual_seed(SEED)\n    model     = MambaSOHPredictor('
    )
    # Move model to device
    code = code.replace(
        'model     = MambaSOHPredictor(input_features=INPUT_FEATURES, feat_dim=SPECTRAL_FEAT_DIM, d_model=D_MODEL, d_state=D_STATE)',
        'model     = MambaSOHPredictor(input_features=INPUT_FEATURES, feat_dim=SPECTRAL_FEAT_DIM, d_model=D_MODEL, d_state=D_STATE).to(device)'
    )
    # Move batches to device in training loop
    code = code.replace(
        '        for X_batch, X_feat_batch, y_batch in train_loader:\n            optimizer.zero_grad()\n            pred',
        '        for X_batch, X_feat_batch, y_batch in train_loader:\n'
        '            X_batch, X_feat_batch, y_batch = X_batch.to(device), X_feat_batch.to(device), y_batch.to(device)\n'
        '            optimizer.zero_grad()\n            pred'
    )
    # Move val/test tensors to device in evaluate calls
    code = code.replace(
        'val_pred = model(X_val, X_feat_val)',
        'val_pred = model(X_val.to(device), X_feat_val.to(device))'
    )
    code = code.replace(
        'test_metrics = evaluate(model, X_test, X_feat_test, y_test)',
        'test_metrics = evaluate(model, X_test.to(device), X_feat_test.to(device), y_test.to(device))'
    )
    # Fix evaluate to handle GPU tensors
    code = code.replace(
        '        pred = model(X, X_feat) * 100.0\n        mae  = torch.mean(torch.abs(pred - y)).item()\n        rmse = torch.sqrt(torch.mean((pred - y) ** 2)).item()',
        '        pred = model(X, X_feat) * 100.0\n        y_dev = y.to(pred.device)\n        mae  = torch.mean(torch.abs(pred - y_dev)).item()\n        rmse = torch.sqrt(torch.mean((pred - y_dev) ** 2)).item()',
    )

    with open(train_path, 'w') as f:
        f.write(code)
    print('GPU support patched into train.py')
else:
    print('train.py already has GPU support')

## Cell 9 — Smoke Test (1 epoch, fast)

Verify no errors before full training.

In [ ]:
os.chdir(REPO)
!python scripts/train.py \
    --data-dir "{PROCESSED}" \
    --epochs 1 \
    --log-dir "{LOGS}/smoke"

print('\nSmoke test done. If no error above, proceed to full training.')

## Cell 10 — Full Training

Estimated time on P100: ~3-5 min/epoch, ~3-5 hours total (early stop at 40-70 epochs).

In [ ]:
os.chdir(REPO)
!python scripts/train.py \
    --data-dir "{PROCESSED}" \
    --epochs 150 \
    --log-dir "{LOGS}"

## Cell 11 — Verify Artifacts

In [ ]:
required = [
    'scaler.pkl',
    'feature_scaler.pkl',
    'soh_mamba_v1.3.pth',
    'isolation_forest_v1.3.pkl',
]

all_ok = True
for fname in required:
    path = f'{WEIGHTS}/{fname}'
    ok   = os.path.isfile(path)
    size = os.path.getsize(path) / 1024 if ok else 0
    print(f'{fname}: {"OK" if ok else "MISSING"}  ({size:.0f} KB)')
    all_ok = all_ok and ok

print()
print('All artifacts ready:', all_ok)

## Cell 12 — Check Training Results

In [ ]:
import glob

log_files = glob.glob(f'{LOGS}/**/train_*.log', recursive=True)
if log_files:
    latest = max(log_files, key=os.path.getmtime)
    print(f'Log: {latest}')
    !grep -E 'Test MAE|Test RMSE|ACHIEVED|WARNING.*target' "{latest}"
else:
    print('No log found')

## Cell 13 — Package for Download

In [ ]:
import shutil, torch

# Check model checkpoint
ckpt_path = f'{WEIGHTS}/soh_mamba_v1.3.pth'
if os.path.isfile(ckpt_path):
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'Model v{ck["version"]}:')
    print(f'  window_size   = {ck["window_size"]}')
    print(f'  input_features= {ck["input_features"]}')
    print(f'  feat_dim      = {ck["feat_dim"]}')
    print(f'  d_model       = {ck.get("d_model", 64)}')
    print(f'  Test MAE      = {ck["test_mae"]}%')
    print(f'  Test RMSE     = {ck["test_rmse"]}%')

# Create zip for download
out_zip = '/kaggle/working/mamba_v1.3_L4096_artifacts'
shutil.make_archive(out_zip, 'zip', WEIGHTS)
zip_size = os.path.getsize(f'{out_zip}.zip') / 1024
print(f'\nCreated: {out_zip}.zip  ({zip_size:.0f} KB)')
print('Download from: Kaggle Output tab -> mamba_v1.3_L4096_artifacts.zip')

## Cell 14 — View Training Curve

In [ ]:
import re, glob
import matplotlib.pyplot as plt

log_files = glob.glob(f'{LOGS}/**/train_*.log', recursive=True)
if not log_files:
    print('No log found')
else:
    latest = max(log_files, key=os.path.getmtime)
    epochs, val_maes, train_losses = [], [], []
    with open(latest) as f:
        for line in f:
            m = re.search(r'DEBUG\s+(\d+)\s+([\d.]+)\s+[\d.]+\s+([\d.]+)', line)
            if m:
                epochs.append(int(m.group(1)))
                train_losses.append(float(m.group(2)))
                val_maes.append(float(m.group(3)))

    if epochs:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(epochs, train_losses, label='Train Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.set_title('Training Loss'); ax1.legend()
        ax2.plot(epochs, val_maes, label='Val MAE%', color='orange')
        ax2.axhline(2.0, color='red', linestyle='--', label='Target 2%')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE %')
        ax2.set_title('Validation MAE'); ax2.legend()
        plt.tight_layout()
        plt.savefig('/kaggle/working/training_curve.png', dpi=150)
        plt.show()
        print(f'Best Val MAE: {min(val_maes):.4f}% at epoch {epochs[val_maes.index(min(val_maes))]}')
    else:
        print('No epoch data in log yet')